# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# 新しく使う人へ —— 全体の地図と、最初の一歩

**高専マップ(KosenMap)を初めて触る人が、最初に上から読むもの。** Web アプリ(本番ホストの Docker)と Android アプリの両方を、
「何があり、誰がどこから入り、毎日どこを触るか」の順に案内する。

**細部はここに書かない。** 手順の本体は各ノートにあり、ここからはリンクで飛ぶ(同じことを2か所に書くと、片方だけ古くなる)。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

> **2026-09-14 の変更は、手元では済んでいるが本番へはまだ配備していない**
> (Mailpit を本番で起動しない・cron 版 6・地図の正本を2つに分ける・phpMyAdmin の root 禁止 など)。
> 下で「※配備後」と付けたものがそれ。いまの本番との差は [status.md](status.md) の §7 にある。

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 1. これは何か

**学校の構内地図と、それを管理する仕組み。** 利用者から見える面は4つある。

| 面 | 入口 | 誰が使う | 認証 |
|---|---|---|---|
| **公開の地図ページ** | `https://ito4.jp/` | 誰でも | 不要(教職員の氏名など一部だけ解除コード) |
| **管理画面** | `https://admin.ito4.jp/admin/` | 管理者 | Logto でサインイン + ロール `kosenmap-admin` + MFA |
| **Android「高専マップ」**(来場者版) | 管理画面「ダウンロード」の APK | 来場者・スタッフ | 地図はアクセスコードで受け取る。ログインは任意 |
| **Android「高専マップ 管理」**(管理版) | 別に配る(下の §5) | 管理者 | Logto でサインイン(管理者ロール) |

**地図の正本は Website(DB)。** 管理版アプリでも直せるが、直したものは管理画面で取り込んでから配る。
配るときは **Website の正本(`kosen-main`)** と **イベント用の正本(`kosen-event`)** の2つがあり、
来場者の端末がどちらを受け取るかは**アクセスコード**で決まる(※配備後。§4)。

### 全体の図

本番は `compose.yaml` に `compose.vps.yaml` を重ねて動く(`.env` の `COMPOSE_FILE`)。
**外から受けるのは reverse-proxy(nginx)だけ**で、ほかは用途ごとに分けた Docker の中のネットワーク(`edge`・`appdb`・`authdb`・`mail`。※12 の配備後。以前は `devnet` 1つ)にいる。

```mermaid
flowchart LR
  user["利用者のブラウザ<br/>Android アプリ"]
  admin["管理者のブラウザ"]
  ext["相手のメールサーバー"]

  subgraph host["本番ホスト Ubuntu Server 24.04 LTS + Docker"]
    nginx["reverse-proxy<br/>km-nginx-proxy<br/>nginx"]
    web["web<br/>km-php-apache<br/>PHP 8.4 + Apache"]
    mariadb[("mariadb<br/>km-mariadb<br/>MariaDB 11.4")]
    postgres[("postgres<br/>km-postgres<br/>PostgreSQL 17")]
    logto["logto<br/>km-logto<br/>Logto 1.43.0"]
    pma["phpmyadmin<br/>km-phpmyadmin"]
    soketi["soketi<br/>km-soketi"]
    mailpit["mailpit<br/>km-mailpit<br/>本番では起動しない"]
    mailserver["mailserver<br/>km-mailserver<br/>送信専用 Postfix"]
    certbot["certbot<br/>km-certbot"]
  end

  user -->|"80 / 443 / 3001 / 6001"| nginx
  admin -->|"3002 / 8281<br/>IP 制限 + Logto のゲート"| nginx
  nginx -->|"443"| web
  nginx -->|"3001 サインイン / 3002 Console"| logto
  nginx -->|"8281"| pma
  nginx -->|"6001 WebSocket"| soketi
  nginx -.->|"8025 はローカル環境だけ"| mailpit
  nginx -.->|"ゲートの判定<br/>/admin/api/gate.php"| web
  web -->|"3306 TLS"| mariadb
  pma -->|"3306 TLS"| mariadb
  logto -->|"5432"| postgres
  web -->|"6001 配信"| soketi
  web -->|"587"| mailserver
  logto -.->|"確認コードのメール"| mailserver
  mailserver -.->|"送信"| ext
  certbot -.->|"証明書を共有ボリュームに置く"| nginx
```

> 図が出ないときは、下の表と [08-architecture](08-architecture.ipynb) §3 の文字の図を見る。
> 校内 LAN の検証構成(`compose.yaml` だけ)は、443 の代わりに 9443、80 の代わりに 8080 で出る。

| ポート | 受けるもの | 公開の理由 |
|---|---|---|
| **80** | Let's Encrypt の確認(`/.well-known/acme-challenge/`)と、HTTPS への転送(308) | 証明書の取得・更新 |
| **443** | 公開ページ・管理画面・Android 向けの API | 本体 |
| **3001** | Logto(サインインの入口) | **ブラウザとアプリが直接ここへ飛ぶ。** 塞ぐと誰もサインインできない |
| **6001** | Soketi(管理画面のチャット・監視のリアルタイム更新) | ブラウザが直接 WebSocket を張る |
| **3002** | Logto Console(ユーザー・ロールの管理) | 管理系。IP 制限 + ゲートの内側 |
| **8281** | phpMyAdmin | 管理系。IP 制限 + ゲートの内側 |
| **8025** | Mailpit の画面 | **本番では publish しない**(2026-09-18)。ローカル環境だけで開く |
| (公開しない) | MariaDB 3306・PostgreSQL 5432・mailserver 587・Soketi の 9601 | 同じネットワークの中からだけ届く |

## 2. 各部品の役割と入口

| サービス名(compose) | コンテナ名 | 役割 | 入口 | 詳しくは |
|---|---|---|---|---|
| `reverse-proxy` | `km-nginx-proxy` | **唯一の入口。** TLS・回数制限・同時接続の上限・管理系のゲート・CSP の予備。設定は `nginx/default.conf.template`(起動時に `KM_DOMAIN` などを埋める)と `nginx/km/` | 80 / 443 / 3001 / 3002 / 6001 / 8281(ローカル環境は + 8025) | [08](08-architecture.ipynb) §3 |
| `web` | `km-php-apache` | PHP 8.4 + Apache。公開ページ・管理画面・Android 向けの `/api/*.php`。`src/` を丸ごと mount | 443 の奥 | [08](08-architecture.ipynb) §2 |
| `mariadb` | `km-mariadb` | **アプリのデータ全部**(地図・イベント・問い合わせ・監査ログ …)。TLS 必須 | 外に出ない。phpMyAdmin か `docker compose exec` | [03](03-backup.ipynb) |
| `postgres` | `km-postgres` | **Logto 専用の DB**(利用者・ロール・設定)。PHP は使わない | 外に出ない | [05](05-containers.ipynb) §5-2 |
| `logto` | `km-logto` | サインイン・MFA・ロール。**管理画面のゲートそのもの**(落ちると管理画面ごと入れない) | 3001(サインイン)/ 3002(Console) | [05](05-containers.ipynb) §5-2 |
| `phpmyadmin` | `km-phpmyadmin` | MariaDB を画面で見る。root でのログインは既定で禁止(※配備後) | 8281 | — |
| `soketi` | `km-soketi` | WebSocket の配信(Pusher 互換)。配信は PHP からだけ行う | 6001 | — |
| `mailpit` | `km-mailpit` | **開発用のメールの受け皿。外へは1通も送らない。** 本番では profile で外してあり、ポートも publish しない | 8025(ローカル環境だけ) | — |
| `mailserver` | `km-mailserver` | **本番の送信専用メールサーバー**(Postfix + DKIM)。問い合わせの通知・定期のお知らせ・Logto の確認コード | 外に出ない(587 は中だけ) | [04](04-host-jobs-and-mail.ipynb) |
| `certbot` | `km-certbot` | Let's Encrypt の証明書を 12 時間ごとに更新。残り 14 日を切ると unhealthy | なし | 下の §6 |

**ボリュームの実体名は `test_` で始まる**(`test_mariadb_data` など)。名前が古いのは承知のうえで、**書き換えると空のボリュームが作られる。**
理由は [08-architecture](08-architecture.ipynb) §7 と `compose.yaml` の末尾。

### 誰がどこから入れるか

**守りは重ねてある。1枚に頼らない。**

| 入口 | 1枚目 | 2枚目 | 3枚目 |
|---|---|---|---|
| 公開ページ `/` | 入口の回数制限(問い合わせ・解除コードの送信など) | — | 氏名などは解除コード(`api/map-unlock.php`、失敗 8 回で一時ロック) |
| 管理画面 `/admin/` | Logto でサインイン | ロール `kosenmap-admin`(スコープ `admin:users:read`。**変更の POST は `admin:users:write` も要る**) | **MFA**(default テナントは Mandatory) |
| Logto サインイン `:3001` | 書き込み(POST)だけ回数制限 | Logto 自身のパスワード規則と試行制限 | MFA |
| Soketi `:6001` | Origin が管理画面(`KM_ADMIN_URL`。別オリジンにしていなければ `KM_APP_URL` と同じ)か空であること | 購読は `admin/api/chat-auth.php` の署名 | — |
| Logto Console `:3002`・phpMyAdmin `:8281`(ローカル環境は Mailpit `:8025` も) | **送信元 IP の制限**(`nginx/km/allow-admin.conf`) | **Logto のゲート**(nginx の `auth_request` が `admin/api/gate.php` に聞く。read と write の両方と、アカウント停止を見る) | それぞれのログイン(Console のアカウント / DB の利用者) |
| Android 向け `/api/*.php`・`/logto_me.php` | 入口の回数制限 | 管理者・スタッフ向けの中身は Bearer トークンで判定 | — |

- **ゲートの判定は `proxy_pass` の手前。** 未ログインの相手には phpMyAdmin も Console も一切渡らない
- 管理系ポートに入れる IP は、`allow-admin.conf` の SSH トンネル用(127.0.0.1・ホスト自身の IP。`172.16.0.0/12` は 2026-09-15 に外した ※12 の配備後)と、
  **ホストにだけ置く** `nginx/km/allow-admin-home.local.conf`(自宅などの固定 IP。見本は同じ名前の `.example`)
- 許可されていない回線から管理系へ入るときは **SSH の `-D`(SOCKS)** を使う(§6)。`-L` はホスト名が変わって Cookie が送られず、ログイン画面を回り続ける
- **Logto Console のアカウント(admin テナント)の MFA は、2026-09-15 の実測で Mandatory。** パスワード方針と総当たりロックを default テナントに揃える手順は [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) §4

## 3. 最初にやること

### 3-1. この PC の準備

**手順の本体は [00-start](00-start.ipynb) §1〜§2。** 要るものだけ挙げる:
VS Code と Jupyter 拡張 / Python 3.13 と ipykernel / PowerShell 7 / `php` / OpenSSL / **SSH の鍵 2 本(`~\.ssh\km_ops` と `~\.ssh\km_vps`)**。

入ったら 00-start の「環境の確認」と「ホストへ繋がるか」を流す。

### 3-2. SSH の鍵を作る

ホストは**鍵でしか入れない**(パスワードのログインは切ってある)。入れる利用者は `km` と `kmops` だけ
(クラウド既定の `ubuntu` は 2026-09-18 に退役させた。[12](12-hardening-2026-09-15.ipynb) §7-4 B)。

**鍵は役ごとに 2 本**:

| 鍵 | 入る利用者 | 何に使うか |
|---|---|---|
| `~\.ssh\km_ops` | `kmops` | 配備・`%%host`・控え(docker を使う。**sudo は無い**) |
| `~\.ssh\km_vps` | `km` | `sudo` が要るセル(`%%terminal`。**docker は無い**) |

1. 下のセル(🔑 別の窓)で 2 本作る(既にあれば作り直さない)。**パスフレーズを付け、ssh-agent に載せる**
2. 出てくる**公開鍵の1行**を、既にホストへ入れる人に渡す。`km_ops` は `kmops` の、`km_vps` は `km` の `authorized_keys` へ
   (`kmops` の方は `sudo scripts/host-ops-user.sh create --pubkey-line "…"` が置く)
3. 00-start の「環境の確認」と「ホストへ繋がるか」が通れば完了

> **パスフレーズ無しにしない。** 以前は「`%%host` は `BatchMode` なので空にする」と書いていたが、
> **その鍵が盗まれるとホストがそのまま取られる。** ssh-agent(Windows のサービス)に載せておけば `BatchMode` でも通る。
> agent の準備は [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) §7-3。
> 週次の控えだけは、パスフレーズを聞けないので `km_backup`(門番付き・控えを作って取る以外は断る)を使う(同 §7-4 A)。

### 鍵を作り、公開鍵を表示する

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%terminal
# 役ごとに 2 本。**パスフレーズは自分で決めて、どこにも書かない**(12 §7-3・§7-4 B)
foreach ($k in 'km_ops', 'km_vps') {
    $key = "$env:USERPROFILE\.ssh\$k"
    if (Test-Path $key) {
        Write-Host "既にあります(作り直しません): $key"
    } else {
        New-Item -ItemType Directory -Force "$env:USERPROFILE\.ssh" | Out-Null
        ssh-keygen -t ed25519 -f $key -C "$k@$env:COMPUTERNAME"
    }
    Get-Content "$key.pub"
}
# agent に載せる(サービスが止まっていたら 12 §7-3 の管理者のセルで自動起動にする)
Get-Service ssh-agent
ssh-add "$env:USERPROFILE\.ssh\km_ops"; ssh-add "$env:USERPROFILE\.ssh\km_vps"; ssh-add -l


### 3-3. 管理者のアカウントとロール

**利用者の正本は Logto。** 管理画面の「ユーザー管理」は読み取り専用の一覧で、追加・停止・ロールの変更は **Logto Console(`:3002`)** で行う。

| 名前 | 種類 | 何ができるか |
|---|---|---|
| `kosenmap-admin` | ロール | 管理画面に入る。`admin:users:read` / `admin:users:write` / `admin:api-keys:read` / `admin:api-keys:write` を持つ |
| `kosenmap-user` | ロール | 一般の利用者 |
| `admin:users:read` | 権限(スコープ) | **管理画面に入れる**(無ければ 403「管理者の権限が割り当てられていません」) |
| `admin:users:write` | 権限(スコープ) | **管理画面での変更(POST)と、3002 / 8281(ローカル環境は 8025 も)のゲート** に要る |
| `staff:event:access` | 権限(スコープ) | **運営スタッフ。** 管理者の権限は無いが、アプリでイベントの通行止めを通過でき、スタッフ限定の地点が見える |

新しい管理者を迎える流れ:

1. **既に管理者の人が** Logto Console でその人のアカウントを用意し、ロール `kosenmap-admin` を割り当てる
2. 本人が `https://admin.ito4.jp/admin/` でサインインする。**初回に MFA(認証アプリなど)の登録を求められる**
3. 管理画面に入れたら、右上のプロフィールで「admin:users:read を含む権限があるため …」と出ることを見る
4. 3002 / 8281 を使う人は、入る回線の IP を `allow-admin-home.local.conf` に足すか、SSH の `-D` を使う

> ロールを割り当てた後は**サインインし直す**(権限はサインインのときのトークンに載る)。

## 4. 毎日の使い方(管理画面)

`https://admin.ito4.jp/admin/`。左のメニューの並びどおりに挙げる。

| メニュー | ファイル | 何をするか |
|---|---|---|
| **ダッシュボード** | `index.php` | 監視対象サービス・登録ユーザー・テーブル数・24時間の操作のタイルと、サービス一覧 |
| ユーザー管理 | `users.php` | Logto の利用者とロールの一覧(**読むだけ**。変えるのは Console) |
| データベース → テーブル管理 | `tables.php` | DB の表を画面で見る・直す |
| データベース → **アプリの地図を取り込む** | `map-sync.php` | 管理版アプリで書き出した地図を Website に取り込む。**適用の前に増える・置き換わる・消える件数を見せる**。既定では消さない |
| データベース → **アプリへ地図を配信する** | `map-publish.php` | 地図を Android へ配る(下の「地図を配る」) |
| データベース → **地図編集** | `map-editor.php` | 地点・経路を直す。`?event=…` 付きで開くとイベントの重ね合わせ(通行止め・臨時の地点・臨時名称)を直す |
| データベース → **イベントモード** | `map-events.php` | イベントの入れ物を作る。**終了予定を過ぎても有効なままのものは先頭で警告**する |
| データベース → 地図データ公開設定 | `map-settings.php` | 公開地図の錠(解除コード)の設定 |
| データベース → phpMyAdmin | `:8281` | 別のタブで開く(IP 制限 + ゲート) |
| サービス監視 | `monitor.php` | コンテナの様子をリアルタイムに(Soketi) |
| コンテンツ → **タイムライン** | `timeline.php` | **監査ログ。** 誰がいつ何を変えたか・拒否・停止で弾いた記録。右上の通知の「直近24時間の操作」もここ |
| コンテンツ → **受信箱** | `mailbox.php` | 公開ページの**問い合わせ**(`/contact.php`)。メールは「気付くため」で、本文の正本はここ |
| コンテンツ → FAQ / 規約とポリシー | `faq.php` / `legal.php` | 公開ページに出る FAQ と、規約・ポリシーへ足す章 |
| コンテンツ → プロジェクト状況 / チャート | `projects.php` / `charts.php` | 作業の状況と集計 |
| **ダウンロード** | `downloads.php` | **配布ファイル。** Android の APK を置く・差し替える(210MB まで)。**アプリへ配信中の地図の期限**もここで見る |
| その他ページ | `profile.php` ほか | プロフィール・カレンダー・かんばん・チャット・ファイル管理 |
| 認証 (Logto) | `:3002` / `:3001` | 管理コンソール・認証エンドポイントを別のタブで開く |
| 設定 | `settings.php` | 画面の設定 |

- **管理画面はかならず `https://admin.ito4.jp` の名前で開く。** 別の名前(IP など)で開くと、チャットと監視が「未接続」になる(6001 の Origin の検査。※配備後)
- 画面にエラーの詳細は出ない。「照合用 ID: xxxxxxxx」が出たら、その ID で web のログを探す(§6 のログのセル。※配備後)

### 地図を配る(アプリへ地図を配信する)

**手順と注意の本体は [07-map-qr](07-map-qr.ipynb)。** ここは全体の流れだけ。※配備後の画面で書いている。

```
地図を直す(地図編集 / 管理版で直したものは「アプリの地図を取り込む」)
   ↓
会期の変更(イベントモード → 地図編集のイベント編集)
   ↓
配る(アプリへ地図を配信する)── Website の正本 kosen-main   … DB から作って配る。イベント画面の内容を折り込む
                              └ イベント用の正本 kosen-event … 管理版アプリの書き出し(JSON、10MB まで)を添付して配る
   ↓
アクセスコードを作る(配信先ごと)→ 画面の QR を印刷する・配る
   ↓
確かめる(ダウンロード画面の「アプリへ配信中の地図」: 正本の種類・版・期限・コードの数)
```

| 決まり | 理由 |
|---|---|
| **有効期限は必ず入れる** | 空のまま配ると、端末に残った地図がいつまでも消えない |
| **アクセスコードは作った直後だけ画面に出る** | URL やログに平文を残さないため。控え損ねたら作り直す |
| **コードが配信先を決める** | `kosen-main` のコードを入れた端末は Website の正本、`kosen-event` のコードならイベント用を受け取る |
| イベント用の「Website のイベント(通行止め)も折り込む」は**既定オフ** | オフならファイルに入っているイベントをそのまま配る |
| 地点が1件も無い JSON は受け付けない | 受け取った全端末で地図が空になるため |

**配信先の無いアクセスコード。** どこにも配っていない名前を指すコードがあると、画面の一番上に黄色のカード
「配信先の無いアクセスコードがあります」が出る。**「付け替え先」で `kosen-main` か `kosen-event` を選んで「付け替える」**か、「止める」を押す。
付け替えてもコードそのものは変わらないので、**印刷して配った QR はそのまま使える。**
本番には `TEST1` を指すコードが1件あり、いまはこのコードで地図を取れない(配備後に付け替える)。

## 5. Android アプリ

リポジトリは `C:\Users\itota\Documents\Test`(この Website とは別)。1つのコードから**2つのアプリ**(フレーバー)を作る。

| | 来場者版 `visitor` | 管理版 `admin` |
|---|---|---|
| アプリ名 | **高専マップ** | **高専マップ 管理** |
| applicationId | `com.ito.kosenmap` | `com.ito.kosenmap.admin`(**1台に両方入れられる**) |
| 地図の出どころ | **サーバーから受け取る**(アクセスコード)。**期限が来ると消える** | **その端末で作った原本。** サーバーから取得せず、期限でも消えない |
| メニュー | 一般向けマップ・ランキング・マップを取得・設定 | 一般向けマップ・ランキング・設定に加え、管理者ロールでログインすると **管理者向けマップ・Wi-Fiアナライザー・インポート / エクスポート** |
| ログイン | 任意。要求する権限は `profile` `email` `staff:event:access` だけ | 管理機能に要る。`admin:*` も要求する |
| 設定の「配信の状態」 | 「版 N / 期限 …」 | 「この端末の編集」(**Website へまだ渡していない編集があるか**) |

> 来場者版で管理機能を押すと「この機能は管理者用アプリ「高専マップ 管理」でご利用ください。」と出る。

### 入れ方

- **来場者版:** 管理画面「ダウンロード」の **Android アプリ (APK)** → 「APK をダウンロード」(`/api/download.php?slug=apk`。**ログイン不要**)。
  端末の設定で「提供元不明のアプリ」を許可してからインストールする
- 「ダウンロード」の APK の枠は**1つだけ**。管理版の APK は、この画面の外で管理者に渡す
- 置き換えるときは同じ画面の下のフォームから差し替える。**サーバーを配備してからアプリを配る**(順番が逆だと、新しいアプリが古いサーバーと話す)

### 地図を受け取る(来場者版)

1. 地図が無いと「アクセスコードを入力するか、QRを読み取ってください。」と出る。メニューの **マップを取得** からも開ける
2. **「QRを読み取る」** で QR を枠に入れる。**読んだ内容は入力欄に入るだけで、送信はしない**(※アプリの次の版から)。
   「QRから読み取りました。内容を確かめて「取得」を押してください。」と出るので、自分で **「取得」** を押す
3. カメラが使えない端末は、コードを手で打てばよい
4. 期限の 7 日前から目立つ表示になり、切れると「マップの有効期限が切れました」「新しいアクセスコードを受け取ってください。」になる

### ログイン

メニューの **ログイン** → ブラウザで Logto(`:3001`)が開く → サインイン(**MFA を登録済みの人はその確認も**)→ アプリへ戻る。

- **スタッフ**(`staff:event:access`)は、来場者版でもイベントの通行止めを通過でき、スタッフ限定の地点が見える。**ログアウトすると、端末の地図からその分を落とす**
- 管理版の管理機能は、管理者ロールでログインするまで出ない

### ランキング

- **参加の既定はオフ**(※アプリの次の版から)。参加するときは説明を読み、**「参加する」を押したときだけ**参加になる
- 参加すると、ログイン中に地図で開いた地点の件数が**表示名と一緒に記録され、ログインしていない人を含め誰でも見られる**
- 参加をやめると、それまでの記録をサーバーから消す。ランキングの画面そのものはログイン無しで見られる

### よくある困りごと(アプリ)

| 出る文言 | 意味 | 打つ手 |
|---|---|---|
| アクセスコードが違います。 | コードが一致しない(HTTP 401) | 打ち間違い・古いコード。管理画面で配信先ごとのコードを確かめる |
| 試行回数が多すぎます。しばらく待ってからやり直してください。 | **外れが 15 分に 30 回に達した**(同じ回線ごと。HTTP 429) | 待つ。学校の回線は1つの IP を大勢で共有するので、誰かの打ち間違いが重なっても出る |
| このマップは配信設定が未完了です。 | コードは正しいが、**その配信先に地図が無い**(配信先の無いコード・未配信) | 管理画面で付け替える / 配信する(§4) |
| マップは最新です。 | 手元の版がサーバーと同じ | そのまま使える |
| マップの有効期限が切れました | 配信の期限を過ぎた | 管理画面で期限を延ばして配り直し、新しいコードを配る |
| LogtoのOIDC設定を取得できません。… | `:3001` に届かない / Logto が落ちている | [06-emergency](06-emergency.ipynb)。Logto が落ちると管理画面にも入れない |
| 証明書の期限切れか、自己署名のサーバーへ繋いでいる可能性があります。 | TLS が通らない | §6 の証明書のセル |
| 配信データが大きすぎます。… | 応答が上限(地図 20MB)を超えた | 配った地図の中身を見直す |
| この機能は管理者用アプリ「高専マップ 管理」でご利用ください。 | 来場者版で管理機能を押した | 管理版を入れる |
| Logtoの管理者ロールが必要です。 | 管理版で、管理者ロールの無いアカウント | Console でロールを割り当て、ログインし直す |

### 開発者向け: 両方のフレーバーを試験してビルドする

**アプリを直したら、来場者版と管理版の両方を試験してビルドするまでが1つの作業。** 片方だけ通って片方が落ちることがある。

- **JDK 21 は PATH に無い。** `JAVA_HOME` を先に置く(下のセルは `C:\Program Files\Eclipse Adoptium` の中から探す)。
  **JDK は更新で場所が変わる。** 変わったら `Test/gradle.properties` の `org.gradle.java.home` も直す
- 件数は `app/build/test-results/*/TEST-*.xml` を数えて見る(`BUILD SUCCESSFUL` だけでは件数が分からない)。
  2026-09-14 の時点で、両フレーバーとも 571 件・失敗 0
- 作った APK は `app\build\outputs\apk\` の下

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps
$jdk = Get-ChildItem 'C:\Program Files\Eclipse Adoptium' -Directory -Filter 'jdk-21*' -ErrorAction SilentlyContinue |
    Sort-Object Name -Descending | Select-Object -First 1
if (-not $jdk) { throw 'JDK 21 が見つかりません(C:\Program Files\Eclipse Adoptium の中を確かめる)' }
$env:JAVA_HOME = $jdk.FullName
"JAVA_HOME = $env:JAVA_HOME"

Set-Location (Join-Path $env:USERPROFILE 'Documents\Test')
.\gradlew.bat testVisitorDebugUnitTest testAdminDebugUnitTest assembleVisitorDebug assembleAdminDebug --console=plain
if ($LASTEXITCODE -ne 0) { exit $LASTEXITCODE }

""
foreach ($task in 'testVisitorDebugUnitTest', 'testAdminDebugUnitTest') {
    $tests = 0; $failures = 0; $errors = 0
    Get-ChildItem "app\build\test-results\$task\TEST-*.xml" | ForEach-Object {
        $suite = ([xml](Get-Content $_.FullName -Raw -Encoding utf8)).testsuite
        $tests += [int]$suite.tests; $failures += [int]$suite.failures; $errors += [int]$suite.errors
    }
    '{0}: 試験 {1} 件 / 失敗 {2} / エラー {3}' -f $task, $tests, $failures, $errors
}
Get-ChildItem app\build\outputs\apk -Recurse -Filter *.apk | Select-Object Name, Length, LastWriteTime

### 開発者向け: 配るもの(リリース)を作る

- **接続先は gradle のプロパティで決まる。** `-Pkosenmap.domain=<ドメイン>`(既定 `ito4.jp`)から、
  サイトの URL(`https://<ドメイン>/`)と Logto(`https://<ドメイン>:3001`)を導く。`https://` やポートを付けるとビルドの時点で止まる
- **`kosenmap.apiResource` はサーバーの `.env` の `KM_API_RESOURCE` と同じ値**(既定 `https://ito4.jp/api`。2026-09-17 にドメインと一緒に移した)。
  変わるとログイン自体が `invalid_target` で失敗する
- 署名の設定(`keystore.properties`)は**リポジトリの外に置き**、パスで渡す。無いとビルドの途中で止まる(無署名の APK を作らない)
- リリースは R8 で縮小される。**試験が全部通っても、配る前に実機で1回、ログインと地図の取得まで確かめる**

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps
$domain   = 'ito4.jp'   # ← ドメインを変えたらここ(https:// もポートも付けない)
$keystore = ''              # ← keystore.properties のパス(リポジトリの外)
if (-not $keystore -or -not (Test-Path $keystore)) { throw '$keystore に keystore.properties のパスを書いてください' }

$jdk = Get-ChildItem 'C:\Program Files\Eclipse Adoptium' -Directory -Filter 'jdk-21*' -ErrorAction SilentlyContinue |
    Sort-Object Name -Descending | Select-Object -First 1
if (-not $jdk) { throw 'JDK 21 が見つかりません' }
$env:JAVA_HOME = $jdk.FullName

Set-Location (Join-Path $env:USERPROFILE 'Documents\Test')
.\gradlew.bat assembleVisitorRelease assembleAdminRelease --no-configuration-cache --console=plain "-Pkosenmap.domain=$domain" "-PkosenmapKeystoreProperties=$keystore"
if ($LASTEXITCODE -ne 0) { exit $LASTEXITCODE }
Get-ChildItem app\build\outputs\apk -Recurse -Filter *release*.apk | ForEach-Object {
    '{0}  {1:N0} バイト  SHA256 {2}' -f $_.Name, $_.Length, (Get-FileHash $_.FullName -Algorithm SHA256).Hash
}

## 6. ホストの基本

| 項目 | 値 |
|---|---|
| ホスト | `ito4.jp`(Ubuntu Server 24.04 LTS、RAM 約 2GB) |
| 入る利用者 | `kmops`(**docker を使える。sudo は無い**)と `km`(**docker は無い**。sudo はパスワードを聞かれる) |
| 置き場 | `/opt/kosenmap`(持ち主は `kmops`。ここで `docker compose …` を叩く) |
| 鍵 | この PC の `~\.ssh\km_ops`(kmops)・`~\.ssh\km_vps`(km)・`~\.ssh\km_backup`(週次の控え。門番付き) |

**コードの配備は [02-deploy](02-deploy.ipynb)(`deploy-to-host.ps1`)。ホストの上で直接ファイルを直さない**(次の配備で上書きされる)。

### SSH で入る

`%%host` のセルで足りるときは、窓を開く必要は無い。対話で作業したいときだけ使う。

管理系のポート(3002 / 8281)に**許可されていない回線から**入るときは、窓で次を実行したまま、
ブラウザの SOCKS5 プロキシを `localhost:1080`(**DNS もプロキシ側で引く**)にして `https://admin.ito4.jp:8281` などを開く:

```powershell
ssh -D 1080 -N -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp
```

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp

### コンテナの様子とメモリ

`docker compose ps` のサービス名とコンテナ名の対応もここで見る。**メモリは上限(`mem_limit`)に張り付いていないか**を見る(上限は ※配備後)。
**healthy は「中の HTTP が返る」ことしか言っていない**([01-daily-check](01-daily-check.ipynb) §2)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose ps --format 'table {{.Service}}\t{{.Name}}\t{{.Status}}'
echo
docker stats --no-stream --format 'table {{.Name}}\t{{.MemUsage}}\t{{.PIDs}}'

### ログを読む

`SERVICE` を見たいサービス名に書き換える。管理画面に「照合用 ID」が出たときは、`web` のログをその ID で探す。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
SERVICE=web   # reverse-proxy / web / logto / mariadb / postgres / soketi / phpmyadmin / mailserver / certbot
ID=''         # 照合用 ID で絞るときだけ書く(8 桁)
if [ -n "$ID" ]; then
  docker compose logs --timestamps --since 72h "$SERVICE" 2>&1 | grep -F "$ID"
else
  docker compose logs --timestamps --tail 60 "$SERVICE"
fi

### 定期処理(cron 版 6)

`/etc/cron.d/kosenmap-updates` は `scripts/host-updates-setup.sh --fix`(root)が書く。**本番は 2026-09-14 の時点でまだ版 5**(配備後に `--fix` で版 6 にする)。
[04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) §1 の表も版 6 に揃えてある(版 5 から**証明書の2行**が増えた)。

| 時刻 | 何を | メール |
|---|---|---|
| 月〜土 2:40 | バックアップ(`host-backup.sh --notify`) | 失敗したときだけ |
| **日曜 2:40** | バックアップ + 実行記録の添付(`--heartbeat --attach-logs`) | **必ず** |
| **毎日 3:47** | **証明書の更新**(`send-log.sh --only-failure --run "host-cert.sh renew"`) | **失敗したときだけ**(版 6 で追加) |
| **毎月1日 4:07** | **証明書の状態**(`host-cert.sh status` を `send-log.sh` で) | **必ず**(版 6 で追加) |
| 毎日 4:17 / **毎月1日 4:23** | コンテナの更新を調べる(`check-updates.sh`) | 変化があったとき / **必ず** |
| 毎日 8:23 / **毎月1日 8:29** | ホストのセキュリティ(`host-security-check.sh`) | 問題があったとき / **必ず** |

**「必ず」の便りが来ない週・月は、仕掛けが止まっている。** 仕込みの状態を見るのは [01-daily-check](01-daily-check.ipynb) の「定期処理の仕込み」。

### 証明書(Let's Encrypt)

証明書は `certbot` のコンテナが 12 時間ごとに、cron(版 6)が毎日 3:47 に更新を試みる。
**HSTS を有効にしてあるので、切らすと誰もサイトに入れなくなる。**

`host-cert.sh status` は、残り日数(21 日を切ると「注」、14 日を切ると ★)・証明書の名前・更新の方式と、
**nginx が 443 で実際に出している証明書が最新と一致するか**を出す。
「更新方式: standalone」の注意が出たら、配備後に `host-cert.sh fix-conf`(まず `--dry-run`)で揃える。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 180
./scripts/host-cert.sh status </dev/null

### ドメインを変えるとき

**本番の `.env` で必須なのは `KM_DOMAIN` だけ**で、URL・証明書のパス・差出人などは `compose.vps.yaml` が導く(※配備後)。
ホストごと移すときは [09-new-host](09-new-host.ipynb)。同じ流れの実行できるセルは 09 の §6「ドメインを変える」にもある。同じホストのままドメインだけ変える流れは次のとおり。

| # | やること | どこで |
|---|---|---|
| 0 | DNS の A レコードを、新しいドメイン → このホストへ | DNS の業者 |
| 1 | 新しいドメインの証明書を取る(**まず `--dry-run`**。発行回数を消費しない) | 下のセル |
| 2 | `host-domain.sh check` で何が変わるかを見る → `apply` で `.env` を書き換える(**控え `.env.bak-<日時>` を作る。自分では up しない**) | 下のセル |
| 3 | `docker compose up -d` | 下のセル |
| 4 | Logto に登録した戻り先の URL を新しいドメインへ(`logto-domain.php`。**まず一覧、それから `--apply`**) | 下のセル |
| 5 | DKIM / SPF / DMARC / 逆引き(PTR)、reCAPTCHA のドメイン、`apply` が最後に出す「次にやること」 | 各業者の画面 |
| 6 | Android を両フレーバーとも作り直して配る(§5 のリリースのセル。`kosenmap.apiResource` は据え置く) | この PC |

- **`KM_API_RESOURCE`(アクセストークンの audience)は変えない。** 変えると配布済みのアプリも管理画面も入れなくなる。`apply` は行が無ければ今の値を書いて固定する
- `apply` は、新しいドメインの証明書が無いときと、配備が古くて導出が効かないときは**書き換える前に止まる**
- 戻すときは `apply` が表示する控えを `.env` に戻して `docker compose up -d`

下のセルの `NEW` / `OLD` / `MAIL` を書き換えてから実行する。

#### 1a. 証明書を試しに取る(`--dry-run`)

DNS がこのホストを指しているか、80 番の確認用の道が外から届くかを見てから、試験用の発行元で通す。
確認のために certbot の置き場へ小さなファイルを1つ書く。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "新しいドメインの証明書を試しに発行します(試験用の発行元。本物の証明書は変えません)" --timeout 600
NEW=kosenmap.example.jp   # ← 新しいドメイン(https:// もポートも付けない)
MAIL=admin@example.jp     # ← 期限切れの警告を Let's Encrypt から受け取る宛先
./scripts/host-cert.sh issue --domain "$NEW" --email "$MAIL" --dry-run </dev/null

#### 1b. 証明書を取る

`letsencrypt` ボリュームに証明書を足す。**`.env` と nginx は変えない。**

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "新しいドメインの証明書を Let's Encrypt から発行します" --timeout 600
NEW=kosenmap.example.jp
MAIL=admin@example.jp
./scripts/host-cert.sh issue --domain "$NEW" --email "$MAIL" </dev/null

#### 2a. 何が変わるかを見る(`host-domain.sh check`)

`.env` は変えない。書き換えた後の形を確かめるため、`/opt/kosenmap` に**一時的な写し(他人は読めない)を作り、終わると消す。**

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
NEW=kosenmap.example.jp
./scripts/host-domain.sh check "$NEW" </dev/null

#### 2b. `.env` を書き換える(`host-domain.sh apply`)

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm ".env の KM_DOMAIN を新しいドメインに書き換えます(控えを作ります。up はしません)" --timeout 300
NEW=kosenmap.example.jp
./scripts/host-domain.sh apply "$NEW" </dev/null

#### 3. 立ち上げ直す

**サイトが数十秒止まる。** 会期中は避ける。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "docker compose up -d で、新しいドメインの設定を読み込ませます" --timeout 900
docker compose up -d
docker compose ps

#### 4a. Logto の戻り先を一覧で見る(`logto-domain.php`)

**Logto の設定は変えない。** Management API のトークンを取るので、web の中の `cache/` にトークンの控えが置かれる(管理画面が普段していることと同じ)。
`-u www-data` を外さない(root で走らせるとスクリプトが止まる)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 180
OLD=ito4.jp   # ← 変える前のドメイン
docker compose exec -T -u www-data web php scripts/logto-domain.php --from="$OLD" </dev/null

#### 4b. Logto の戻り先を書き換える

リダイレクト URI・サインアウト後の URI・CORS・webhook などのうち、**ホスト名が旧ドメインと完全に一致するものだけ**を換える。
アプリのカスタムスキーム(`com.ito.kosenmap.auth://…`)と API リソースには触れない。メールのテンプレートの中の URL は Console で目で確かめる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto に登録した戻り先の URL を新しいドメインへ書き換えます" --timeout 180
OLD=ito4.jp
docker compose exec -T -u www-data web php scripts/logto-domain.php --from="$OLD" --apply </dev/null

### バックアップと復元

**本体は [03-backup](03-backup.ipynb)。** 覚えておくことだけ:

- ホストは**毎晩 2:40** に `/opt/kosenmap/backups/` へ控えを取り、**日曜は「取れています」のメールが必ず届く**
- この PC は**週次のタスク**でホストの控えを取り寄せる(03 §5)。**控えは実際に開いてみるまで信用しない**(03 §4)
- 戻すのは 03 §8。**`POSTGRES_USER` は名前まで同じにする**
- **`docker compose down -v` は使わない**(ボリュームごと消える)

## 7. 困ったとき

**まず [06-emergency](06-emergency.ipynb) の「まず1本」(`host-emergency.sh`)。** 何も直さず・止めず、様子と復旧の手順を並べる。
**「とりあえず再起動」は、何が起きていたかを消す。** 見てから打つ手を選ぶ。

### よくある誤解

| 誤解 | 本当は |
|---|---|
| 本番のメールは Mailpit で見られる | **Mailpit は本番に居ない。** 本番の送信は `mailserver`。届いたかは送信サーバーの記録の `status=sent` で見る([01](01-daily-check.ipynb) §2)。**8025 は 2026-09-18 から本番では開いていない** |
| `/contact.php/abc` のような URL が 404 になるのは壊れている | **わざと。** `.php` の後ろにパスを足した URL(PATH_INFO)は nginx と Apache の両方で断っている(※配備後)。このサイトは PATH_INFO を使っていない |
| 429 はサーバーの不調 | **回数制限。** 入口の nginx(同時接続の上限・問い合わせ・サインイン・API・アプリの地図の取得)と、アプリのコードの外れ(15 分に 30 回)で返す。**待てば戻る。** 学校の回線は1つの IP を大勢で共有するので、教室でいっせいに出るようなら数字を見直す(`nginx/default.conf.template`) |
| 413 は壊れている | 本文の大きさの上限。公開ページと API は 3MB、管理画面は 12MB、APK の差し替えは 210MB |
| healthy なら使える | healthy は「中の HTTP が返る」だけ。利用者の経路は画面で見る([05](05-containers.ipynb) §7) |
| 管理画面の「ユーザー管理」で人を足せる | 読むだけ。**足す・止める・ロールを変えるのは Logto Console** |
| ロールを付けたのに入れない | **サインインし直す。** 権限はサインインのときのトークンに載る |
| 管理系のポートへは `ssh -L` で入れる | **`-L` はログイン画面を回り続ける。** `-D`(SOCKS)を使う(§6) |
| 3306 に外から繋げばよい | **MariaDB は外に出していない。** phpMyAdmin(8281)か `docker compose exec` |
| アクセスコードは後で画面から見られる | **作った直後だけ。** 控え損ねたら作り直す |
| 期限は空でもよい | **必ず入れる。** 空だと端末に残った地図が消えない |
| ホストの上でファイルを直せば済む | 次の配備で上書きされる。手元で直して [02-deploy](02-deploy.ipynb)。例外は `.env`・`config/*.local.php`・`nginx/km/*.local.conf`(ホストが正本) |
| 管理画面は IP でも名前でも同じ | 名前(`https://admin.ito4.jp`)で開く。別の名前ではチャットと監視が「未接続」になる(※配備後) |

落とし穴の一覧は [08-architecture](08-architecture.ipynb) §8 にもある。

## 8. 用語集

| 用語 | 意味 |
|---|---|
| **KosenMap / 高専マップ** | この仕組み全体。公開の地図ページ・管理画面・Android アプリ |
| **本番** | `ito4.jp`(`/opt/kosenmap`)。検証用の受け皿は無い |
| **正本** | 食い違ったときに正しいとする側。**利用者は Logto、地図は Website(DB)、`.env` はホスト** |
| **`kosen-main`** | Website の正本から作って配る地図の配信 ID |
| **`kosen-event`** | 管理版アプリの書き出し(JSON)を添付して配る、イベント用の地図の配信 ID |
| **アクセスコード** | 来場者の端末が地図を受け取るための合言葉。**どの配信 ID を受け取るかも決める**。サーバーにはハッシュだけを残す |
| **配信先の無いコード** | どの配信 ID も指していないコード。付け替えるか止める |
| **版(revision)/ 有効期限** | 配った地図の番号と、端末で使える期限。期限を過ぎた地図は来場者版から消える |
| **イベントモード** | 通行止め・臨時の地点・臨時名称を、期間を決めて地図に重ねる仕組み |
| **解除コード(公開地図の錠)** | 公開ページで教職員の氏名などを見るための合言葉。外れ 8 回で一時ロック |
| **フレーバー** | 1つのコードから作り分けるアプリの種類。`visitor`(高専マップ)と `admin`(高専マップ 管理) |
| **Logto** | サインイン・MFA・ロールを担う認証基盤。3001 がサインイン、3002 が Console |
| **テナント** | Logto の区画。`default` がこのサイトの利用者、`admin` が Console のアカウント |
| **ロール / スコープ(権限)** | `kosenmap-admin` が `admin:users:read` などを束ねる。スタッフは `staff:event:access` |
| **MFA** | パスワードに加える2つ目の確認(認証アプリなど)。default テナントは必須 |
| **API リソース(`KM_API_RESOURCE`)** | アクセストークンの宛先名(audience)。**URL の形をした名前で、ドメインを変えても変えない** |
| **ゲート** | 管理系ポートの手前で nginx が `admin/api/gate.php` に「通してよいか」を聞く仕組み(`auth_request`) |
| **`allow-admin.conf`** | 管理系ポートに入れる送信元 IP。個人の固定 IP はホストの `allow-admin-home.local.conf` |
| **`KM_DOMAIN`** | 本番の `.env` で必須の、サイトのドメイン。ほかの URL はここから導く |
| **`compose.vps.yaml`** | 校内 LAN 用の `compose.yaml` に重ねる、インターネットに出すための差分 |
| **profile** | compose で「指定したときだけ起動する」印。本番の Mailpit に付けてある |
| **HSTS** | 「このサイトは HTTPS だけ」とブラウザに覚えさせる仕組み。**証明書を切らすと誰も入れない**理由 |
| **DKIM / SPF / DMARC / PTR** | 送ったメールが迷惑メール扱いされないための DNS の設定。ドメインを変えたら全部直す |
| **heartbeat** | 問題が無くても送る定期の便り。**来ないこと**で仕掛けの停止に気づく |
| **監査ログ(タイムライン)** | 管理画面での操作・拒否の記録 |
| **照合用 ID** | 画面に出す 8 桁の印。詳細は画面に出さず、この ID でログを探す |
| **`check.php`** | サーバー側の自己検査は1本(`php src/scripts/check.php`)。DB も Logto も要らない |
| **`deploy-to-host.ps1`** | この PC から本番へコードを送るスクリプト([02](02-deploy.ipynb)) |
| **`host-cert.sh` / `host-domain.sh` / `logto-domain.php`** | 証明書を見る・更新する・取る / ドメインを変える / Logto の戻り先を換える |
| **`%%ps` / `%%host` / `%%terminal`** | このノートのセルの型。この PC の PowerShell / ホストの sh / 別の窓([00-start](00-start.ipynb) §4) |
| **🟢 🟡 🔴 🔑** | セルの印。読むだけ / 手元が変わる / 本番が変わる / 別の窓 |